In [0]:
### Business Profile

#### VyomNet Telecom is one of India's leading telecom operators serving more than 22 telecom circles across the country. Every day the company receives millions of Call Detail Records (CDRs), recharge transactions, and subscriber updates from multiple operational systems.

## The Analytics team prepares monthly reports for senior leadership and regulatory authorities. During the previous quarter, the Compliance team discovered that different departments were calculating Monthly Active Users (MAU) using different methodologies. Additionally, duplicate CDR records caused subscriber counts to fluctuate between reports.

### Problem Statement: The Head of Analytics has asked the Data Engineering team to rebuild the analytics layer so that every business report follows a single, standardized methodology. You have been assigned as the Senior Data Engineer responsible for designing clean and reliable datasets that will be consumed by downstream reporting teams.

In [0]:
#### Initializing the spark session
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql.types import*
spark= SparkSession.builder.appName("vyomnet").getOrCreate()

In [0]:
### Reading the datasets from the volume and building the dataframes

call_records_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/3_vyomnettelecom/raw_data/call_records.csv")

circle_reference_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/3_vyomnettelecom/raw_data/circle_reference.csv")

recharge_transaction_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/3_vyomnettelecom/raw_data/recharge_transactions.csv")

subscribers_df= spark.read.format("csv").option("header","true").option("mode","PERMISSIVE").option("inferSchema","false").load("/Volumes/coding_playbook/3_vyomnettelecom/raw_data/subscribers.csv")

In [0]:
#### TASK 1: Performing the data cleaning and transformation steps on the respective datasets
call_records_df.printSchema()

### As we can see that all the columns are respectively in string data type for which we have to perform the right schema enforcement
call_records_df_1= call_records_df.select(f.col("cdr_id"),f.col("subscriber_id"),f.col("call_timestamp").alias("timestamp"),f.trim(f.initcap(f.col("call_type"))).alias("call_type"),f.col("duration_seconds").cast("int"),f.col("tower_circle"),f.trim(f.upper(f.col("network_type"))).alias("network_type"),f.col("ingestion_batch"))
call_records_df_1.show(5)
call_records_df_1.printSchema()

### Filtering the good records and the bad records and also dropping the duplicate records based on the subset of cdr_id, subscriber_id and timestamp, and call type

call_records_df_final= call_records_df_1.filter((f.col("cdr_id").isNotNull())&(f.col("duration_seconds")>=0)&(f.col("subscriber_id").isNotNull()))

call_records_df_final=call_records_df_final.dropDuplicates(["cdr_id","timestamp","subscriber_id","call_type"])

call_records_df_quarantine= call_records_df_1.filter((f.col("cdr_id").isNull())|(f.col("duration_seconds")<0)|(f.col("subscriber_id").isNull())|(f.col("timestamp").isNull()))

### After we are done with filtering our good and bad records we then have to segregate them under different directories

call_records_df_final.write.format("delta").mode("overwrite").option("mergeSchema","true").save("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/call_records_good/")

call_records_df_quarantine.write.format("delta").mode("overwrite").option("mergeSchema","true").save("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_quarantine_data/call_records_bad/")

In [0]:
### Data Transformation on the Recharge Transactions Datasets

### As we can see that currently all the columns are of datatype string and we have to enforce the correct datatypes
recharge_transaction_df_1= recharge_transaction_df.select(f.col("txn_id"),f.col("subscriber_id"),f.col("txn_timestamp").cast("timestamp"),f.col("amount").cast("int"),f.trim(f.col("channel")).alias("channel"),f.col("payment_status"))

recharge_transactions_df_1=recharge_transaction_df_1.dropDuplicates(["txn_id","subscriber_id","txn_timestamp"])

recharge_transactions_df_1.show(5)
recharge_transactions_df_1.printSchema()

#### After dropping the duplicates we now hav to quarantine the bad records and seperate the good records
recharge_transactions_final=recharge_transactions_df_1.filter((f.col("txn_id").isNotNull())&(f.col("subscriber_id").isNotNull())&(f.col("txn_timestamp").isNotNull()))

recharge_transactions_quarantine=recharge_transactions_df_1.filter((f.col("txn_id").isNull())|(f.col("subscriber_id").isNull())|(f.col("txn_timestamp").isNull()))

### Now its time to seperately write these records in different locations
recharge_transactions_final.write.format("delta").option("mergeSchema","true").save("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/recharge_transactions_good/")

recharge_transactions_quarantine.write.format("delta").option("mergeSchema","true").save("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_quarantine_data/recharge_transactions_bad/")

In [0]:
### Lastly we have to perform the data cleaning and data pre-processing of the subscribers datasets. 

subscribers_df_final= subscribers_df.dropDuplicates(["subscriber_id","msisdn","circle"])
subscribers_df_final.write.format("delta").mode("overwrite").save("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/subscribers_good/")


In [0]:
###### DELIVERABLE 1: Calculate the Monthly Active Users (MAU) for every telecom circle. A subscriber should be considered active if they have made at least one call during a month. Expected Output MontH Circle Active Subscribers

call_records_df_req= spark.read.format("delta").load("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/call_records_good/")

call_records_df_req= call_records_df_req.withColumn("month_year",f.date_format(f.col("timestamp"),"yyyy-MM"))

mau_df=call_records_df_req.select(f.col("cdr_id"),f.col("subscriber_id"),f.col("month_year"),f.col("tower_circle")).orderBy(f.col("subscriber_id").asc(),f.col("month_year").asc())

##### We first have to find the total count of subscribers who should be marked active if they have placed at least once call during a month
from pyspark.sql.window import Window
window_created= Window.partitionBy(f.col("subscriber_id"),f.col("month_year")).orderBy(f.col("month_year").asc())
mau_df= mau_df.withColumn("active_score",f.rank().over(window_created))

#### Momthly active users for each month
mau_active_users_1= mau_df.filter(f.col("active_score")==1)
mau_active_users_final= mau_active_users_1.groupBy(f.col("tower_circle"),f.col("month_year")).agg(f.count("*").alias("total_active_users")).orderBy(f.col("tower_circle").asc(),f.col("month_year").asc())

mau_active_users_final.show()

In [0]:
##### DELIVERABLE 2: For every subscriber, calculate a running total of recharge amount over time. Output should contain: subscriber_id,txn_timestamp, amount, cumulative_recharge.

recharge_transactions_req= spark.read.format("delta").load("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/recharge_transactions_good/")

recharge_transactions_req= recharge_transactions_req.withColumn("month_year",f.date_format(f.col("txn_timestamp"),"yyyy-MM"))
recharge_transactions_req_1= recharge_transactions_req.select(f.col("subscriber_id"),f.col("month_year"),f.col("amount"))

from pyspark.sql.window import Window

window_created_3= Window.partitionBy(f.col("subscriber_id")).orderBy(f.col("month_year")).rowsBetween(Window.unboundedPreceding, Window.currentRow)

running_total_df=recharge_transactions_req_1.groupBy(f.col("subscriber_id"),f.col("month_year"),f.col("amount")).agg(f.sum(f.col("amount")).over(window_created_3).alias("running_recharge_total"))

running_total_df.show()

In [0]:
#### DELIVERABLE 3: Calculate subscriber tenure based on call activity. Find: First Call Date, Last Call Date, Tenure in Days. Then Sort subscribers from highest tenure to lowest tenure.

from pyspark.sql.window import Window
window_created_4= Window.partitionBy(f.col("subscriber_id")).orderBy(f.col("timestamp").asc())

req_call_records_df= spark.read.format("delta").load("/Volumes/coding_playbook/3_vyomnettelecom/final_data/bronze_datasets/bronze_good_data/call_records_good/").filter(f.col("call_type")==f.lit("Voice")).select(f.col("subscriber_id"),f.col("duration_seconds"),f.col("timestamp").cast("date"))

req_call_records_df= req_call_records_df.withColumn("first_call_date",f.min(f.col("timestamp")).over(window_created_4))
req_call_records_df= req_call_records_df.withColumn("latest_call_date",f.max(f.col("timestamp")).over(window_created_4))

req_call_records_df=req_call_records_df.withColumn("tenure_days",f.abs(f.datediff(f.col("first_call_date"),f.col("latest_call_date")))).orderBy(f.col("tenure_days").desc())

req_call_records_df= req_call_records_df.select(f.col("subscriber_id"),f.col("first_call_date"),f.col("latest_call_date"),f.col("tenure_days"))

req_call_records_df.show(5)